# 02 — Image Processing

Preprocessing pipeline:

```
CDL (USA-wide, EPSG:5070, 10 m)
  └─ Step 1: Reproject EPSG:5070 → EPSG:4326  +  clip to study area  +  resample to S2 res
  └─ Step 2: Filter CDL classes (keep target crops only)

S2 images (EPSG:4326, ~10 m, study area)

Verify: CDL and S2 share the same CRS, bounds, and pixel grid
```

> **Why CDL first?**  The CDL raster covers the entire USA.  
> Reprojecting + clipping it to the study area *before* any S2 work keeps all subsequent operations cheap.

## Configuration

In [ ]:
import sys, os
from glob import glob

sys.path.append("../")

# ── S2 source directory (Google Drive mount) ──────────────────────────────────
S2_DIR = "/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/S2_Annual_15d_sacramento_5"

# 2024 only
S2_YEAR = "2024"
S2_ALL = sorted(glob(os.path.join(S2_DIR, f"S2H_{S2_YEAR}_*.tif")))

print(f"S2 files found ({S2_YEAR}): {len(S2_ALL)}")

# ── CDL label (external drive) ────────────────────────────────────────────────
CDL_RAW = {
    "2024": "/Volumes/T7/research-crop-mapping-geoai/data/cdl/2024_10m_cdls/2024_10m_cdls.tif",
}

# ── Output directories (external drive) ───────────────────────────────────────
OUT_DIR = "/Volumes/T7/research-crop-mapping-geoai/data/processed"
OUT_CDL_DIR = os.path.join(OUT_DIR, "cdl")
OUT_S2_DIR = os.path.join(OUT_DIR, "s2")
os.makedirs(OUT_CDL_DIR, exist_ok=True)
os.makedirs(OUT_S2_DIR, exist_ok=True)

# Per-year CDL output paths
CDL_REPROJECTED = {yr: os.path.join(OUT_CDL_DIR, f"cdl_{yr}_study_area.tif") for yr in CDL_RAW}
CDL_FILTERED = {
    yr: os.path.join(OUT_CDL_DIR, f"cdl_{yr}_study_area_filtered.tif") for yr in CDL_RAW
}

# ── Parameters ────────────────────────────────────────────────────────────────
S2_NODATA_VALUE = -9999.0
CDL_NODATA_VALUE = 0

# 8 active crop classes (CalCROP21 >=1M-px threshold). Fallow/Idle (61) → background.
KEEP_CLASSES = [1, 3, 24, 36, 54, 69, 75, 76]
#   1=Corn, 3=Rice, 24=Winter Wheat, 36=Alfalfa,
#   54=Tomatoes, 69=Grapes, 75=Almonds, 76=Walnuts

print(f"\nKEEP_CLASSES ({len(KEEP_CLASSES)}): {KEEP_CLASSES}")
print(f"OUT_DIR: {OUT_DIR}")
for yr, raw in CDL_RAW.items():
    print(f"CDL {yr}: {'OK' if os.path.exists(raw) else 'MISSING'} {raw}")

S2 files found (2024): 25

KEEP_CLASSES (8): [1, 3, 24, 36, 54, 69, 75, 76]
OUT_DIR: /Volumes/T7/research-crop-mapping-geoai/data/processed
CDL 2024: OK /Volumes/T7/research-crop-mapping-geoai/data/cdl/2024_30m_cdls/2024_30m_cdls.tif


---
## Inspect: CRS Mismatch
Before processing, confirm the coordinate system difference between CDL and S2.

In [2]:
import rasterio

print("=" * 55)
with rasterio.open(S2_ALL[0]) as src:
    s2_crs = src.crs
    s2_bounds = src.bounds
    s2_res = src.res
    s2_shape = (src.width, src.height)
    s2_bands = src.count
    print(f"S2  ({os.path.basename(S2_ALL[0])})")
    print(f"  CRS       : {s2_crs}")
    print(f"  Bounds    : {s2_bounds}")
    print(f"  Resolution: {s2_res}")
    print(f"  Size      : {s2_shape[0]} x {s2_shape[1]}  |  {s2_bands} bands")

print()
with rasterio.open(CDL_RAW["2024"]) as src:
    cdl_crs = src.crs
    cdl_bounds = src.bounds
    cdl_res = src.res
    cdl_shape = (src.width, src.height)
    print(f"CDL ({os.path.basename(CDL_RAW['2024'])})")
    print(f"  CRS       : {cdl_crs}")
    print(f"  Bounds    : {cdl_bounds}")
    print(f"  Resolution: {cdl_res} m")
    print(f"  Size      : {cdl_shape[0]} x {cdl_shape[1]}")

print()
print(f"CRS match?  {'Yes' if s2_crs == cdl_crs else 'No — reprojection required'}")
print("=" * 55)

S2  (S2H_2024_2024_01_01.tif)
  CRS       : EPSG:4326
  Bounds    : BoundingBox(left=-122.07161480135971, bottom=38.69511052650522, right=-121.46453333235175, top=39.1609768328496)
  Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
  Size      : 6758 x 5186  |  11 bands

CDL (2024_30m_cdls.tif)
  CRS       : EPSG:5070
  Bounds    : BoundingBox(left=-2356095.0, bottom=276915.0, right=2258235.0, top=3172605.0)
  Resolution: (30.0, 30.0) m
  Size      : 153811 x 96523

CRS match?  No — reprojection required


---
## Step 1 — Reproject, Clip & Resample CDL

Three operations in **one pass** using the S2 image as the reference grid:

| | CDL before | CDL after |
|---|---|---|
| CRS | EPSG:5070 (Conus Albers) | EPSG:4326 (WGS84) |
| Coverage | Entire USA | Study area only |
| Resolution | 10 m | ~10 m (matches S2) |

Using `Resampling.nearest` to preserve integer class labels.

In [3]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# Use any S2 file as the reference grid (all S2 share the same grid/CRS/bounds)
S2_REF = S2_ALL[0]
with rasterio.open(S2_REF) as s2_ref:
    target_crs = s2_ref.crs
    target_transform = s2_ref.transform
    target_width = s2_ref.width
    target_height = s2_ref.height

print(f"Reference grid: {target_width}×{target_height}  CRS={target_crs}\n")

for yr, cdl_raw_path in CDL_RAW.items():
    out_path = CDL_REPROJECTED[yr]
    if os.path.exists(out_path):
        print(f"[{yr}] Already exists: {os.path.basename(out_path)}")
        continue
    if not os.path.exists(cdl_raw_path):
        print(f"[{yr}] CDL not found: {cdl_raw_path}  — SKIPPING")
        continue

    with rasterio.open(cdl_raw_path) as cdl_src:
        dst_data = np.zeros((1, target_height, target_width), dtype=np.uint8)
        reproject(
            source=rasterio.band(cdl_src, 1),
            destination=dst_data,
            src_transform=cdl_src.transform,
            src_crs=cdl_src.crs,
            dst_transform=target_transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest,
        )

    profile = {
        "driver": "GTiff",
        "dtype": "uint8",
        "nodata": 0,
        "width": target_width,
        "height": target_height,
        "count": 1,
        "crs": target_crs,
        "transform": target_transform,
        "compress": "lzw",
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(dst_data)
    print(f"[{yr}] Reprojected CDL → {os.path.basename(out_path)}")

Reference grid: 6758×5186  CRS=EPSG:4326

[2024] Reprojected CDL → cdl_2024_study_area.tif


---
## Step 2 — Filter CDL Classes
Keep only the target crop classes; all other pixels are set to 0 (background).

In [4]:
from utils import label as label_utils
from utils.constants import USDA_CDL_NAMES

for yr in CDL_RAW:
    in_path = CDL_REPROJECTED[yr]
    out_path = CDL_FILTERED[yr]

    if not os.path.exists(in_path):
        print(f"[{yr}] Reprojected CDL not found — run Step 1 first")
        continue
    if os.path.exists(out_path):
        print(f"[{yr}] Already filtered: {os.path.basename(out_path)}")
        continue

    label_utils.label_filtering(in_path=in_path, out_path=out_path, keep_classes=KEEP_CLASSES)

    with rasterio.open(out_path) as src:
        data = src.read(1)
        unique, counts = np.unique(data, return_counts=True)
        total = data.size
        print(f"\n[{yr}] Class distribution in filtered CDL:")
        for cls, cnt in zip(unique, counts):
            if cls == 0:
                continue
            name = USDA_CDL_NAMES.get(int(cls), "unknown")
            print(f"  {cls:>4}  {name:<30}  {cnt:>12,}  {cnt/total*100:>5.2f}%")

Saved filtered raster: /Volumes/T7/research-crop-mapping-geoai/data/processed/cdl/cdl_2024_study_area_filtered.tif

[2024] Class distribution in filtered CDL:
     1  Corn                                 868,680   2.48%
     3  Rice                               8,518,512  24.31%
    24  Winter Wheat                       1,179,978   3.37%
    36  Alfalfa                              584,880   1.67%
    54  Tomatoes                           2,081,043   5.94%
    69  Grapes                               679,326   1.94%
    75  Almonds                            3,991,946  11.39%
    76  Walnuts                            2,453,427   7.00%


---
## Verify Alignment
CDL and each S2 image must share the same CRS, bounds, and pixel dimensions.

In [7]:
# Verify a processed S2 sample aligns with the filtered CDL
print(f"{'File':<50}  {'Shape':<14}  {'CDL match'}")
print("-" * 80)

cdl_path = CDL_FILTERED["2024"]
if not S2_ALL or not os.path.exists(cdl_path):
    print("missing files — run Steps 1-2 first")
else:
    with rasterio.open(cdl_path) as cdl:
        cdl_shape = (cdl.width, cdl.height)
        cdl_crs = cdl.crs

    with rasterio.open(S2_ALL[0]) as src:
        ok = (src.width, src.height) == cdl_shape and src.crs == cdl_crs
        print(
            f"[{S2_YEAR}] {os.path.basename(S2_ALL[0]):<48}  {src.width}×{src.height:<6}  {'OK' if ok else 'MISMATCH'}"
        )

print("Alignment check complete.")


File                                                Shape           CDL match
--------------------------------------------------------------------------------
[2024] S2H_2024_2024_01_01.tif                           6758×5186    OK
Alignment check complete.
